# Tokenization :)

### Step 1

Write the `BasicTokenizer` class, with the following three core functions:

- `def train(self, text, vocab_size, verbose=False)`
- `def encode(self, text)`
- `def decode(self, ids)`

Train your tokenizer on whatever text you like and visualize the merged tokens. Do they look reasonable? One default test you may wish to use is the text file `tests/taylorswift.txt`.

In [10]:
def get_stats(ids):
    """Count how often each adjacent pair (ids[i], ids[i+1]) appears.
    Returns dict: (int, int) -> int"""
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx):
    """Replace every consecutive occurrence of `pair` in `ids` with the new token `idx`.
    Returns a new list of ids."""
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [12]:
class BasicTokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {}
        self.vocab_size = 0

    def train(self, text, vocab_size, verbose=False):
        self.vocab_size = vocab_size
        self.vocab = {idx: bytes([idx]) for idx in range(256)}
        self.merges = {}  # (int, int) -> int

        tokens = text.encode("utf-8")
        tokens = list(map(int, tokens))
        num_merges = vocab_size - 256
        ids = list(tokens)  # copy so we don't destroy the original list
        for i in range(num_merges):
            stats = get_stats(ids)
            pair = max(stats, key=stats.get)
            idx = 256 + i

            if verbose:
                print(f"merging {pair} into a new token {idx}")

            ids = merge(ids, pair, idx)
            self.merges[pair] = idx
            self.vocab[idx] = self.vocab[pair[0]] + self.vocab[pair[1]]

    def decode(self, ids):
        tokens = b"".join(self.vocab[idx] for idx in ids)
        text = tokens.decode("utf-8", errors="replace")
        return text

    def encode(self, text):
        # given a string, return list of integers (the tokens)
        tokens = list(text.encode("utf-8"))
        while len(tokens) >= 2:
            stats = get_stats(tokens)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break  # nothing else can be merged
            idx = self.merges[pair]
            tokens = merge(tokens, pair, idx)
        return tokens

In [13]:
# load training text (adjust the path if needed)
with open("../minbpe/tests/taylorswift.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [14]:
tokenizer = BasicTokenizer()
tokenizer.train(text, vocab_size=276, verbose=True)
print("vocab size:", tokenizer.vocab_size)

merging (101, 32) into a new token 256
merging (44, 32) into a new token 257
merging (100, 32) into a new token 258
merging (46, 32) into a new token 259
merging (114, 32) into a new token 260
merging (50, 48) into a new token 261
merging (115, 32) into a new token 262
merging (105, 110) into a new token 263
merging (111, 110) into a new token 264
merging (114, 105) into a new token 265
merging (116, 32) into a new token 266
merging (116, 104) into a new token 267
merging (101, 258) into a new token 268
merging (257, 261) into a new token 269
merging (97, 110) into a new token 270
merging (97, 114) into a new token 271
merging (101, 260) into a new token 272
merging (121, 32) into a new token 273
merging (97, 108) into a new token 274
merging (267, 256) into a new token 275
vocab size: 276


In [15]:
for i in range(256, tokenizer.vocab_size):
    print(i, tokenizer.vocab[i])

256 b'e '
257 b', '
258 b'd '
259 b'. '
260 b'r '
261 b'20'
262 b's '
263 b'in'
264 b'on'
265 b'ri'
266 b't '
267 b'th'
268 b'ed '
269 b', 20'
270 b'an'
271 b'ar'
272 b'er '
273 b'y '
274 b'al'
275 b'the '


In [16]:
sample = "안녕하세요 👋 (hello in Korean!)"
ids = tokenizer.encode(sample)
ok = tokenizer.decode(ids) == sample
print(ok)

True


### Step 2

Convert you `BasicTokenizer` into a `RegexTokenizer`, which takes a regex pattern and splits the text exactly as GPT-4 would. Process the parts separately as before, then concatenate the results. Retrain your tokenizer and compare the results before and after. You should see that you will now have no tokens that go across categories (numbers, letters, punctuation, more than one whitespace). Use the GPT-4 pattern:

```
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
```

In [17]:
import regex as re

GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""


class RegexTokenizer(BasicTokenizer):
    def __init__(self, pattern=GPT4_SPLIT_PATTERN):
        super().__init__()
        self.pattern = re.compile(pattern)

    def train(self, text, vocab_size, verbose=False):
        self.vocab_size = vocab_size
        self.vocab = {idx: bytes([idx]) for idx in range(256)}
        self.merges = {}

        tokenstrs = re.findall(self.pattern, text)
        ids = [b for s in tokenstrs for b in map(int, s.encode("utf-8"))]
        num_merges = vocab_size - 256
        for i in range(num_merges):
            stats = get_stats(ids)
            pair = max(stats, key=stats.get)
            idx = 256 + i

            if verbose:
                print(f"merging {pair} into a new token {idx}")

            ids = merge(ids, pair, idx)
            self.merges[pair] = idx
            self.vocab[idx] = self.vocab[pair[0]] + self.vocab[pair[1]]

    def encode(self, text):
        tokenstrs = re.findall(self.pattern, text)
        return [b for s in tokenstrs for b in super().encode(s)]

In [18]:
re_tokenizer = RegexTokenizer()
re_tokenizer.train(text, vocab_size=276, verbose=True)

merging (101, 32) into a new token 256
merging (44, 32) into a new token 257
merging (100, 32) into a new token 258
merging (46, 32) into a new token 259
merging (114, 32) into a new token 260
merging (50, 48) into a new token 261
merging (115, 32) into a new token 262
merging (105, 110) into a new token 263
merging (111, 110) into a new token 264
merging (114, 105) into a new token 265
merging (116, 32) into a new token 266
merging (116, 104) into a new token 267
merging (101, 258) into a new token 268
merging (257, 261) into a new token 269
merging (97, 110) into a new token 270
merging (97, 114) into a new token 271
merging (101, 260) into a new token 272
merging (121, 32) into a new token 273
merging (97, 108) into a new token 274
merging (267, 256) into a new token 275


In [19]:
for i in range(256, re_tokenizer.vocab_size):
    print(i, re_tokenizer.vocab[i])

256 b'e '
257 b', '
258 b'd '
259 b'. '
260 b'r '
261 b'20'
262 b's '
263 b'in'
264 b'on'
265 b'ri'
266 b't '
267 b'th'
268 b'ed '
269 b', 20'
270 b'an'
271 b'ar'
272 b'er '
273 b'y '
274 b'al'
275 b'the '


In [20]:
sample = "안녕하세요 👋 (hello in Korean!)"
ids = re_tokenizer.encode(sample)
ok = re_tokenizer.decode(ids) == sample
print(ok)

True


### Step 3

You're now ready to load the merges from the GPT-4 tokenizer and show that your tokenizer produces the identical results for both `encode` and `decode`, matching [tiktoken](https://github.com/openai/tiktoken).

```
# match this
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
text = enc.decode(ids) # get the same text back
```

Unfortunately, you will run into two issues:

1. It is not trivial to recover the raw merges from the GPT-4 tokenizer. You can easily recover what we call `vocab` here, and what they call and store under `enc._mergeable_ranks`. Feel free to copy paste the `recover_merges` function in `minbpe/gpt4.py`, which takes these ranks and returns the raw merges. If you wish to know how this function works, read [this](https://github.com/openai/tiktoken/issues/60) and [this](https://github.com/karpathy/minbpe/issues/11#issuecomment-1950805306). Basically, under some conditions it is enough to only store the parent nodes (and their rank) and get rid of the precise details of which children merged up to any parent.
2. Second, the GPT-4 tokenizer for some reason permutes its raw bytes. It stores this permutation in the first 256 elements of the mergeable ranks, so you can recover this byte shuffle relatively simply as `byte_shuffle = {i: enc._mergeable_ranks[bytes([i])] for i in range(256)}`. In both your encode and decode, you'll have to shuffle bytes around accordingly. If you're stuck, reference the minbpe/gpt4.py` file for hints.

In [21]:
# helper functions from minbpe
def bpe(mergeable_ranks, token, max_rank):
    # helper function used in get_gpt4_merges() to reconstruct the merge forest
    parts = [bytes([b]) for b in token]
    while True:
        min_idx = None
        min_rank = None
        for i, pair in enumerate(zip(parts[:-1], parts[1:])):
            rank = mergeable_ranks.get(pair[0] + pair[1])
            if rank is not None and (min_rank is None or rank < min_rank):
                min_idx = i
                min_rank = rank
        if min_rank is None or (max_rank is not None and min_rank >= max_rank):
            break
        assert min_idx is not None
        parts = parts[:min_idx] + [parts[min_idx] + parts[min_idx + 1]] + parts[min_idx + 2 :]
    return parts


def recover_merges(mergeable_ranks):
    merges = {}
    for token, rank in mergeable_ranks.items():
        if len(token) == 1:
            continue  # skip raw bytes
        pair = tuple(bpe(mergeable_ranks, token, max_rank=rank))
        assert len(pair) == 2
        # recover the integer ranks of the pair
        ix0 = mergeable_ranks[pair[0]]
        ix1 = mergeable_ranks[pair[1]]
        merges[(ix0, ix1)] = rank

    return merges

In [22]:
import tiktoken


class GPT4Tokenizer(RegexTokenizer):
    def __init__(self):
        super().__init__()
        enc = tiktoken.get_encoding("cl100k_base")  # this is the GPT-4 tokenizer
        mergeable_ranks = enc._mergeable_ranks
        self.merges = recover_merges(mergeable_ranks)
        self.vocab = {rank: token for token, rank in mergeable_ranks.items()}
        self.byte_shuffle = {i: enc._mergeable_ranks[bytes([i])] for i in range(256)}

    def encode(self, text):
        ids = []
        tokenstrs = re.findall(self.pattern, text)
        for piece in tokenstrs:
            piece_ids = [self.byte_shuffle[b] for b in piece.encode("utf-8")]  # shuffle FIRST
            while len(piece_ids) >= 2:  # then BPE
                stats = get_stats(piece_ids)
                pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
                if pair not in self.merges:
                    break
                piece_ids = merge(piece_ids, pair, self.merges[pair])
            ids.extend(piece_ids)
        return ids

In [23]:
enc = tiktoken.get_encoding("cl100k_base")
mergeable_ranks = enc._mergeable_ranks
merges = recover_merges(mergeable_ranks)

In [24]:
len(merges)

100000

In [25]:
s = "hello world!!!? (안녕하세요!) lol123 😉"

enc = tiktoken.get_encoding("cl100k_base")
gpt4tokenizer = GPT4Tokenizer()

gpt4tokenizer.encode(s) == enc.encode(s)

True

In [26]:
gpt4tokenizer.encode(text) == enc.encode(text)

True

### Step 4

(Optional, irritating, not obviously useful) Add the ability to handle special tokens. You'll then be able to match the output of tiktoken even when special tokens are present, e.g.:

```
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("<|endoftext|>hello world", allowed_special="all")
```

Without `allowed_special` tiktoken will error.

In [27]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
ids = enc.encode("<|endoftext|>hello world", allowed_special="all")
ids

[100257, 15339, 1917]

In [28]:
GPT4_SPECIALT_TOKENS = {
    "<|endoftext|>": 100257,
    "<|fim_prefix|>": 100258,
    "<|fim_middle|>": 100259,
    "<|fim_suffix|>": 100260,
    "<|endofprompt|>": 100276,
}

In [33]:
class GPT4Tokenizer(RegexTokenizer):
    def __init__(self):
        super().__init__()
        enc = tiktoken.get_encoding("cl100k_base")  # this is the GPT-4 tokenizer
        mergeable_ranks = enc._mergeable_ranks

        self.special_tokens = GPT4_SPECIALT_TOKENS
        self.merges = recover_merges(mergeable_ranks)
        self.vocab = {rank: token for token, rank in mergeable_ranks.items()}
        self.vocab |= {i: s.encode("utf-8") for s, i in self.special_tokens.items()}
        self.byte_shuffle = {i: enc._mergeable_ranks[bytes([i])] for i in range(256)}

    def encode(self, text, allowed_special=()):
        allowed = set(self.special_tokens) if allowed_special == "all" else set(allowed_special)
        pat = re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
        ids = []
        for piece in pat.split(text):  # split on specials first
            if piece in self.special_tokens:
                if piece not in allowed:
                    raise ValueError(f"disallowed special token {piece!r}")
                ids.append(self.special_tokens[piece])  # emit id directly
                continue

            for sub in self.pattern.findall(piece):
                piece_ids = [self.byte_shuffle[b] for b in sub.encode("utf-8")]
                while len(piece_ids) >= 2:
                    stats = get_stats(piece_ids)
                    pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
                    if pair not in self.merges:
                        break
                    piece_ids = merge(piece_ids, pair, self.merges[pair])
                ids.extend(piece_ids)
        return ids

In [39]:
enc = tiktoken.get_encoding("cl100k_base")
tok = GPT4Tokenizer()  # your class

s = "<|fim_prefix|>hello world"
# encode (allow specials so the last test works)
a = tok.encode(s, allowed_special="all")
b = enc.encode(s, allowed_special="all")
enc_match = a == b

# decode round-trip
dec_match = tok.decode(a) == enc.decode(b) == s

print(enc_match)
print(dec_match)

True
True


### Step 5

If you've made it this far, you're now a pro at LLM Tokenization! Sadly, you're not exactly done yet because a lot of LLMs outside of OpenAI (e.g. Llama, Mistral) use [sentencepiece](https://github.com/google/sentencepiece) instead. Primary difference being that sentencepiece runs BPE directly on Unicode code points instead of on UTF-8 encoded bytes. Feel free to explore sentencepiece on your own (good luck, it's not too pretty), and stretch goal if you really experience and suffer from the burden of time, re-write your BPE to be on Unicode code points and match the Llama 2 tokenizer.
